# Row 2: the decider with the cues in its prompt

What this tests and why: `docs/RATIONALE.md` (section 4, row 2; decision D21). Same model and same question as row 1, with one change:
the three cues from row 3 are written into the prompt. Everything else is held fixed, so any difference is due to the cues.

It reuses the cues already written (row 3's cue file). Only the decider runs: about 10 minutes for the regular 2 s ticks on val, and about
8 more for the moments where the matcher fires (so row 4's combination can be redone with the better-informed decider).

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":      # run from the repo root
    os.chdir("..")
import glob, torch
print(os.getcwd(), "| cuda:", torch.cuda.is_available())
CUES    = sorted(glob.glob("results/cache/cues/*/cues.jsonl"))[-1]
MATCHER = sorted(glob.glob("results/cache/matcher/Ops-MM-embedding-v1-2B__*"))[-1]
ROW1    = sorted(glob.glob("results/cache/decider/Qwen3-VL-2B-Instruct__query_only__*"))[-1]
print("cues:", CUES); print("matcher:", MATCHER); print("row 1 decider:", ROW1)

c:\Users\abhin\Code\CV_Proj | cuda: True
cues: results/cache/cues\Qwen3-VL-2B-Instruct__b055d726\cues.jsonl
matcher: results/cache/matcher\Ops-MM-embedding-v1-2B__875248e9
row 1 decider: results/cache/decider\Qwen3-VL-2B-Instruct__query_only__1a7df950


## 1. Smoke test: two of the shortest val questions, with cues

In [2]:
!python scripts/run_decider_cache.py --samples data/processed/ovo_far_val.jsonl --cues {CUES} --limit 2 --shortest --set decider.variant=query_cues

cache: results\cache\decider\Qwen3-VL-2B-Instruct__query_cues__b1de186d
[1/2] ovo_REC_1566 REC: ran, 10 ticks
[2/2] ovo_REC_1591 REC: ran, 11 ticks
done: 21 model calls in 7 s



Loading weights: 100%|██████████| 625/625 [00:02<00:00, 248.46it/s]
c:\Users\abhin\Code\CV_Proj\.venv\Lib\site-packages\transformers\models\qwen2_vl\image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


## 2. Speed check

Should be close to row 1's 300 ms (the prompt is a little longer).

In [3]:
import glob, json, statistics
rows = [json.loads(l) for f in glob.glob("results/cache/decider/*query_cues*/ovo_*.jsonl") for l in open(f) if l.strip()]
print(len(rows), "decisions; median", round(statistics.median(r["ms"] for r in rows)), "ms")

21 decisions; median 235 ms


## 3. All val questions, regular 2 s ticks (resumable)

In [4]:
!python scripts/run_decider_cache.py --samples data/processed/ovo_far_val.jsonl --cues {CUES} --set decider.variant=query_cues

cache: results\cache\decider\Qwen3-VL-2B-Instruct__query_cues__b1de186d
[1/43] ovo_CRR_1479 CRR: ran, 45 ticks
[2/43] ovo_CRR_1480 CRR: ran, 20 ticks
[3/43] ovo_CRR_1481 CRR: ran, 18 ticks
[4/43] ovo_CRR_1482 CRR: ran, 17 ticks
[5/43] ovo_CRR_1483 CRR: ran, 51 ticks
[6/43] ovo_CRR_1484 CRR: ran, 20 ticks
[7/43] ovo_CRR_1485 CRR: ran, 18 ticks
[8/43] ovo_SSR_1518 SSR: ran, 66 ticks
[9/43] ovo_SSR_1519 SSR: ran, 65 ticks
[10/43] ovo_SSR_1531 SSR: ran, 64 ticks
[11/43] ovo_SSR_1533 SSR: ran, 79 ticks
[12/43] ovo_SSR_1538 SSR: ran, 269 ticks
[13/43] ovo_SSR_1540 SSR: ran, 79 ticks
[14/43] ovo_SSR_1545 SSR: ran, 96 ticks
[15/43] ovo_SSR_1546 SSR: ran, 34 ticks
[16/43] ovo_SSR_1548 SSR: ran, 74 ticks
[17/43] ovo_SSR_1554 SSR: ran, 103 ticks
[18/43] ovo_SSR_1556 SSR: ran, 86 ticks
[19/43] ovo_SSR_1557 SSR: ran, 67 ticks
[20/43] ovo_REC_1558 REC: ran, 16 ticks
[21/43] ovo_REC_1561 REC: ran, 17 ticks
[22/43] ovo_REC_1562 REC: ran, 17 ticks
[23/43] ovo_REC_1564 REC: ran, 17 ticks
[24/43] ovo_REC


Loading weights: 100%|██████████| 625/625 [00:01<00:00, 453.66it/s]
c:\Users\abhin\Code\CV_Proj\.venv\Lib\site-packages\transformers\models\qwen2_vl\image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


## 4. Row 1 against row 2, same questions, same moments

For each question, how well p(Yes) separates moments near an event from the rest (AUC; 0.5 is no skill), averaged over questions,
and the change from adding the cues with a 95% interval that resamples whole videos. **A gain counts only if the interval excludes zero.**

In [5]:
ROW2 = sorted(glob.glob("results/cache/decider/Qwen3-VL-2B-Instruct__query_cues__*"))[-1]
print("row 2 decider:", ROW2)
!python scripts/compare_deciders.py --a {ROW1} --b {ROW2} --samples data/processed/ovo_far_val.jsonl

row 2 decider: results/cache/decider\Qwen3-VL-2B-Instruct__query_cues__b1de186d

the regular 2 s ticks: 43 questions with both kinds of moment in both caches
group                            n   AUC A   AUC B   B - A         95% interval  Yes-rate A     B
all                             43   0.614   0.618   0.003 [-0.038,  0.041]        0.80  0.76
REC / thumos                    10   0.699   0.735   0.036 [ 0.003,  0.068]        0.80  0.91  <- excludes 0
REC / perception_test           14   0.618   0.627   0.009 [-0.055,  0.084]        0.96  0.80
SSR / COIN                      12   0.604   0.525  -0.080 [-0.162, -0.016]        0.73  0.66  <- excludes 0
CRR / MovieNet                   7   0.504   0.592   0.088 [-0.004,  0.131]        0.61  0.64

AUC 0.5 = no skill. 'B - A' is the change from adding what B has; the interval resamples whole videos (several questions share one).


## 5. Row 2 on its own, scored like row 1

Average F1 over thresholds and cooldowns. Compare with row 1's best (32.0, tuned on val) and the timer floors.

In [6]:
!python scripts/replay_decider.py --cache {ROW2} --samples data/processed/ovo_far_val.jsonl

ovo_far_val: scores stored for 43/43 questions, T=1, tol=2s
    tau cooldown    REC    SSR    CRR  avg F1  detections
   0.05      0.0   33.1    8.9    8.5   16.84        1977
   0.10      0.0   33.2    9.1    8.5   16.93        1962
   0.20      0.0   33.4    9.4    8.9   17.24        1887
   0.30      0.0   32.2   14.2    7.4   17.93        1759
   0.40      0.0   33.4    9.3   10.2   17.64        1614
   0.50      0.0   35.0   10.4    6.3   17.22        1516
   0.60      0.0   37.5    7.9    8.7   18.02        1123
   0.70      0.0   35.9    9.9    4.5   16.79         745
   0.80      0.0   38.6    6.3   12.5   19.15         399
   0.90      0.0   18.0    1.4    0.0    6.47         175
   0.95      0.0    3.9    2.6    0.0    2.14          28
   0.05      4.0   47.0   16.9   16.2   26.71        1000
   0.10      4.0   47.1   17.2   16.2   26.83         994
   0.20      4.0   47.3   16.7   16.6   26.87         965
   0.30      4.0   46.9   19.9   13.6   26.79         917
   0.40     

## 6. The decider with cues at the matcher's trigger moments (about 1,800 calls)

This redoes row 4 with the cues in the decider's prompt. It also fills the gaps where a trigger does not fall on a regular tick.

In [7]:
!python scripts/run_gated_decider.py --samples data/processed/ovo_far_val.jsonl --matcher {MATCHER} --cues {CUES} --set decider.variant=query_cues

decider cache: results\cache\decider\Qwen3-VL-2B-Instruct__query_cues__b1de186d
43 questions, 1841 trigger moments to ask about (rule code, any cue, theta >= 0.02)
[1/43] ovo_CRR_1479 CRR: 22 moments, 19 new decider calls
[2/43] ovo_CRR_1480 CRR: 9 moments, 6 new decider calls
[3/43] ovo_CRR_1481 CRR: 15 moments, 9 new decider calls
[4/43] ovo_CRR_1482 CRR: 31 moments, 24 new decider calls
[5/43] ovo_CRR_1483 CRR: 64 moments, 48 new decider calls
[6/43] ovo_CRR_1484 CRR: 18 moments, 14 new decider calls
[7/43] ovo_CRR_1485 CRR: 16 moments, 12 new decider calls
[8/43] ovo_SSR_1518 SSR: 78 moments, 59 new decider calls
[9/43] ovo_SSR_1519 SSR: 103 moments, 77 new decider calls
[10/43] ovo_SSR_1531 SSR: 61 moments, 44 new decider calls
[11/43] ovo_SSR_1533 SSR: 103 moments, 76 new decider calls
[12/43] ovo_SSR_1538 SSR: 296 moments, 231 new decider calls
[13/43] ovo_SSR_1540 SSR: 42 moments, 31 new decider calls
[14/43] ovo_SSR_1545 SSR: 97 moments, 72 new decider calls
[15/43] ovo_SSR_15


Loading weights: 100%|██████████| 625/625 [00:01<00:00, 315.05it/s]
c:\Users\abhin\Code\CV_Proj\.venv\Lib\site-packages\transformers\models\qwen2_vl\image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


In [8]:
!python scripts/compare_deciders.py --a {ROW1} --b {ROW2} --samples data/processed/ovo_far_val.jsonl --matcher {MATCHER}
!python scripts/replay_gated.py --matcher {MATCHER} --decider {ROW2} --samples data/processed/ovo_far_val.jsonl


the regular 2 s ticks: 43 questions with both kinds of moment in both caches
group                            n   AUC A   AUC B   B - A         95% interval  Yes-rate A     B
all                             43   0.614   0.618   0.003 [-0.038,  0.041]        0.80  0.76
REC / thumos                    10   0.699   0.735   0.036 [ 0.003,  0.068]        0.80  0.91  <- excludes 0
REC / perception_test           14   0.618   0.627   0.009 [-0.055,  0.084]        0.96  0.80
SSR / COIN                      12   0.604   0.525  -0.080 [-0.162, -0.016]        0.73  0.66  <- excludes 0
CRR / MovieNet                   7   0.504   0.592   0.088 [-0.004,  0.131]        0.61  0.64

matcher trigger moments (theta 0.02): 38 questions with both kinds of moment in both caches
group                            n   AUC A   AUC B   B - A         95% interval  Yes-rate A     B
all                             38   0.619   0.569  -0.050 [-0.123,  0.015]        0.86  0.82
REC / thumos                     9   0.

## 7. Push the results

```
git add results/cache && git commit -m "Row 2 decider with cues, val" && git pull --rebase origin abhi && git push origin abhi
```